# UMUD — Gray55-Trained Apo Model Eval (Phase 3)

**GPU notebook** — compare **baseline apo model** vs **gray55-trained apo model** on all test images.

Inference for both uses matched preprocessing: gray55 outside bbox + mask clip to bbox.

Outputs:
- `/kaggle/working/apo_gray55_train_eval.csv`
- `/kaggle/working/apo_gray55_train_eval_summary.json`



## Configuration


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

IMG_SIZE = 256
APO_REGION_THRESHOLD = 0.50
GRAY_FILL_VALUE = 55
ROI_THRESH = 5
ROI_PAD_PX = 10


def resolve_pkl(name: str) -> Path:
    preferred = [
        Path(f"/kaggle/input/notebooks/ucheozoemena/umud-train-apo-gray55-phase-3/{name}"),
        Path(f"/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/{name.replace('gray55_', '')}"),
    ]
    for p in preferred:
        if p.exists():
            return p
    hits = sorted(Path("/kaggle/input").rglob(name))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find {name} under /kaggle/input — add train kernel in sidebar")


BASELINE_MODEL = resolve_pkl("apo_baseline.pkl")
GRAY55_MODEL = resolve_pkl("apo_gray55_baseline.pkl")
print("Baseline model:", BASELINE_MODEL)
print("Gray55 model:", GRAY55_MODEL)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"


In [ ]:
from __future__ import annotations

import cv2
import numpy as np
import pandas as pd
from fastai.vision.all import PILImage, load_learner
from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}


def list_test_images(directory: Path) -> list[Path]:
    files = [
        p
        for p in directory.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.name != "Thumbs.db"
    ]
    # Prefer .tif when both .tif and .png exist for same stem
    by_stem: dict[str, Path] = {}
    for p in sorted(files):
        stem = p.stem
        if stem not in by_stem or p.suffix.lower() in {".tif", ".tiff"}:
            by_stem[stem] = p
    return sorted(by_stem.values(), key=lambda p: p.name)


def load_gray(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return arr.astype(np.uint8)


def resize_image(img: np.ndarray, size: int) -> np.ndarray:
    return np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def resize_mask_to(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return (mask > 0).astype(np.uint8)
    src = (mask > 0).astype(np.uint8) * 255
    out = Image.fromarray(src).resize((target_w, target_h), Image.NEAREST)
    return (np.array(out) > 0).astype(np.uint8)


def tensor_to_mask(pred) -> np.ndarray:
    if hasattr(pred, "cpu"):
        pred = pred.cpu().numpy()
    arr = np.asarray(pred)
    if arr.ndim == 3:
        arr = arr.argmax(axis=0)
    return (arr > 0).astype(np.uint8)


def open_rgb_256(img_native: np.ndarray) -> PILImage:
    small = resize_image(img_native, IMG_SIZE)
    rgb = np.stack([small, small, small], axis=-1).astype(np.uint8)
    return PILImage.create(rgb)


def tag_apo_style(coverage: float) -> str:
    return "region" if coverage >= APO_REGION_THRESHOLD else "line"


def invert_mask(mask: np.ndarray) -> np.ndarray:
    return (1 - mask).astype(np.uint8)


def effective_apo_mask(mask: np.ndarray, style: str) -> tuple[np.ndarray, str]:
    if style == "region":
        return invert_mask(mask), "inverted_region"
    return mask, "raw_line"


def find_apo_contours(mask: np.ndarray, min_area_frac: float = 0.0003) -> list[np.ndarray]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = mask.size * min_area_frac
    big = [c for c in contours if cv2.contourArea(c) >= min_area]
    big.sort(key=lambda c: cv2.boundingRect(c)[1])
    return big


def pick_superficial_deep(contours: list[np.ndarray], min_sep_px: int = 15):
    if len(contours) < 2:
        return None, None, len(contours)
    sup = contours[0]
    _, y0, _, _ = cv2.boundingRect(sup)
    deep = None
    for c in contours[1:]:
        _, y1, _, _ = cv2.boundingRect(c)
        if y1 >= y0 + min_sep_px:
            deep = c
            break
    if deep is None:
        deep = contours[min(2, len(contours) - 1)]
    return sup, deep, len(contours)


def edge_polyline(contour: np.ndarray, which: str = "bottom", n_bins: int = 60):
    pts = contour.reshape(-1, 2)
    if len(pts) < 3:
        return np.array([]), np.array([])
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    if x_max <= x_min:
        return pts[:, 0].astype(float), pts[:, 1].astype(float)
    edges = np.linspace(x_min, x_max, n_bins + 1)
    xs_out, ys_out = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = pts[(pts[:, 0] >= lo) & (pts[:, 0] < hi)]
        if len(in_bin) == 0:
            continue
        y = in_bin[:, 1].max() if which == "bottom" else in_bin[:, 1].min()
        xs_out.append((lo + hi) / 2.0)
        ys_out.append(float(y))
    return np.array(xs_out), np.array(ys_out)


def fit_line(xs: np.ndarray, ys: np.ndarray):
    if len(xs) < 2:
        return None
    return np.poly1d(np.polyfit(xs, ys, 1))


def line_angle_deg(line) -> float:
    return float(np.degrees(np.arctan(line[1]))) if line is not None else np.nan


def mt_from_apo_edges(sup_line, deep_line, x_left: float, x_right: float) -> float:
    if sup_line is None or deep_line is None or x_right <= x_left:
        return np.nan
    thirds = [x_left + (x_right - x_left) * t for t in (1 / 6, 3 / 6, 5 / 6)]
    dists = [abs(float(deep_line(x) - sup_line(x))) for x in thirds]
    return float(np.mean(dists))


def fascicle_pca(mask: np.ndarray) -> dict | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return None
    coords = np.column_stack([xs.astype(float), ys.astype(float)])
    centered = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    projections = centered @ direction
    return {
        "length_px": float(projections.max() - projections.min()),
        "angle_deg": float(np.degrees(np.arctan2(direction[1], direction[0]))),
    }


def acute_angle_deg(a1: float, a2: float) -> float:
    d = abs(a1 - a2) % 180.0
    return float(min(d, 180.0 - d))


def apo_geometry_attempt(apo_mask: np.ndarray, style: str) -> dict:
    eff, method = effective_apo_mask(apo_mask, style)
    contours = find_apo_contours(eff)
    sup_c, deep_c, n_contours = pick_superficial_deep(contours)
    out = {
        "apo_method": method,
        "n_contours": n_contours,
        "mt_px": np.nan,
        "deep_angle_deg": np.nan,
        "mt_fail_reason": "ok",
        "sup_line": None,
        "deep_line": None,
        "sup_xs": None,
        "sup_ys": None,
        "deep_xs": None,
        "deep_ys": None,
    }
    if len(contours) == 0:
        out["mt_fail_reason"] = "no_contours"
        return out
    if n_contours < 2 or sup_c is None or deep_c is None:
        out["mt_fail_reason"] = "single_contour"
        return out
    sup_x, sup_y = edge_polyline(sup_c, which="bottom")
    deep_x, deep_y = edge_polyline(deep_c, which="top")
    sup_line = fit_line(sup_x, sup_y)
    deep_line = fit_line(deep_x, deep_y)
    out.update(sup_line=sup_line, deep_line=deep_line, sup_xs=sup_x, sup_ys=sup_y, deep_xs=deep_x, deep_ys=deep_y)
    if sup_line is None or deep_line is None:
        out["mt_fail_reason"] = "line_fit_fail"
        return out
    if len(sup_x) == 0 or len(deep_x) == 0:
        out["mt_fail_reason"] = "empty_edge_polyline"
        return out
    x_left = max(sup_x.min(), deep_x.min())
    x_right = min(sup_x.max(), deep_x.max())
    if x_right <= x_left:
        out["mt_fail_reason"] = "no_x_overlap"
        return out
    out["mt_px"] = mt_from_apo_edges(sup_line, deep_line, x_left, x_right)
    out["deep_angle_deg"] = line_angle_deg(deep_line)
    if np.isnan(out["mt_px"]):
        out["mt_fail_reason"] = "mt_compute_nan"
    return out


def apo_geometry_from_mask(apo_mask: np.ndarray, style: str) -> dict:
    primary = apo_geometry_attempt(apo_mask, style)
    primary["geometry_path"] = primary["apo_method"]
    if not np.isnan(primary["mt_px"]):
        return primary
    if style == "region":
        fallback = apo_geometry_attempt(apo_mask, "line")
        if not np.isnan(fallback["mt_px"]):
            fallback["apo_method"] = f"{primary['apo_method']}+fallback_line"
            fallback["geometry_path"] = "fallback_line"
            fallback["mt_fail_reason_primary"] = primary["mt_fail_reason"]
            return fallback
    primary["mt_fail_reason_primary"] = primary["mt_fail_reason"]
    return primary


def derive_geometry(fasc_mask: np.ndarray, apo_mask: np.ndarray, apo_style: str) -> dict:
    apo = apo_geometry_from_mask(apo_mask, apo_style)
    fpca = fascicle_pca(fasc_mask)
    out = {
        "pa_deg": np.nan,
        "fl_px": np.nan,
        "mt_px": apo["mt_px"],
        "apo_method": apo.get("apo_method"),
        "geometry_path": apo.get("geometry_path"),
        "n_contours": apo.get("n_contours"),
        "mt_fail_reason": apo.get("mt_fail_reason"),
        "mt_fail_reason_primary": apo.get("mt_fail_reason_primary"),
        "apo_cov": float(apo_mask.mean()),
        "apo_fg_pixels": int(apo_mask.sum()),
        "fasc_cov": float(fasc_mask.mean()),
        "fasc_fg_pixels": int(fasc_mask.sum()),
    }
    if fpca is not None:
        out["fl_px"] = fpca["length_px"]
        ref = apo["deep_angle_deg"] if not np.isnan(apo["deep_angle_deg"]) else 0.0
        out["pa_deg"] = acute_angle_deg(fpca["angle_deg"], ref)
    return out


In [ ]:
import cv2
from fastai.vision.all import load_learner

def find_roi_bbox(img_gray: np.ndarray, thr: float = ROI_THRESH, pad: int = ROI_PAD_PX):
    roi = (img_gray > thr).astype(np.uint8)
    if roi.sum() == 0:
        h, w = img_gray.shape
        return 0, h, 0, w
    num, labels, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
    best = max(range(1, num), key=lambda i: stats[i, cv2.CC_STAT_AREA])
    x, y, w, h = stats[best, :4]
    y0 = max(0, y - pad)
    y1 = min(img_gray.shape[0], y + h + pad)
    x0 = max(0, x - pad)
    x1 = min(img_gray.shape[1], x + w + pad)
    return int(y0), int(y1), int(x0), int(x1)


def preprocess_gray55(img_native: np.ndarray):
    bbox = find_roi_bbox(img_native)
    y0, y1, x0, x1 = bbox
    pre = img_native.copy()
    outside = np.ones(img_native.shape, dtype=bool)
    outside[y0:y1, x0:x1] = False
    pre[outside] = GRAY_FILL_VALUE
    return pre, bbox


def clip_mask_to_bbox(mask: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    y0, y1, x0, x1 = bbox
    out = np.zeros_like(mask, dtype=np.uint8)
    out[y0:y1, x0:x1] = mask[y0:y1, x0:x1]
    return out


def infer_apo(learn, img_gray: np.ndarray, bbox, clip_bbox: bool = True):
    h, w = img_gray.shape
    pil = open_rgb_256(img_gray)
    _, apo_t, _ = learn.predict(pil)
    mask = resize_mask_to(tensor_to_mask(apo_t), h, w)
    if clip_bbox:
        mask = clip_mask_to_bbox(mask, bbox)
    style = tag_apo_style(float(mask.mean()))
    geo = apo_geometry_from_mask(mask, style)
    return mask, style, geo, float(mask.mean())

base_learn = load_learner(BASELINE_MODEL)
gray_learn = load_learner(GRAY55_MODEL)
print("Models loaded")


In [ ]:
rows = []
for p in tqdm(sorted(TEST_DIR.glob("*.tif")), desc="eval"):
    with __import__("PIL").Image.open(p) as im:
        arr = np.array(im)
    img = arr.mean(axis=-1).astype(np.uint8) if arr.ndim == 3 else arr.astype(np.uint8)
    h, w = img.shape
    img_g, bbox = preprocess_gray55(img)

    b_mask, b_style, b_geo, b_cov = infer_apo(base_learn, img_g, bbox)
    g_mask, g_style, g_geo, g_cov = infer_apo(gray_learn, img_g, bbox)

    rows.append({
        "image_id": p.name,
        "res": f"{h}x{w}",
        "base_pred_cov": b_cov,
        "gray_model_pred_cov": g_cov,
        "base_style": b_style,
        "gray_model_style": g_style,
        "base_mt_ok": bool(np.isfinite(b_geo["mt_px"])),
        "gray_model_mt_ok": bool(np.isfinite(g_geo["mt_px"])),
        "base_mt_fail_reason": b_geo["mt_fail_reason"],
        "gray_model_mt_fail_reason": g_geo["mt_fail_reason"],
        "base_mt_px": b_geo["mt_px"],
        "gray_model_mt_px": g_geo["mt_px"],
    })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/apo_gray55_train_eval.csv", index=False)

summary = {
    "n_test": len(df),
    "baseline_gray55_infer_mt_ok_mean": float(df.base_mt_ok.mean()),
    "gray55_model_gray55_infer_mt_ok_mean": float(df.gray_model_mt_ok.mean()),
    "baseline_fail_counts": df.loc[~df.base_mt_ok, "base_mt_fail_reason"].value_counts().to_dict(),
    "gray_model_fail_counts": df.loc[~df.gray_model_mt_ok, "gray_model_mt_fail_reason"].value_counts().to_dict(),
    "fixed_by_gray_model": int(((~df.base_mt_ok) & (df.gray_model_mt_ok)).sum()),
    "broken_by_gray_model": int((df.base_mt_ok & ~df.gray_model_mt_ok).sum()),
    "no_contours_baseline": int((df.base_mt_fail_reason == "no_contours").sum()),
    "no_contours_gray_model": int((df.gray_model_mt_fail_reason == "no_contours").sum()),
    "single_contour_baseline": int((df.base_mt_fail_reason == "single_contour").sum()),
    "single_contour_gray_model": int((df.gray_model_mt_fail_reason == "single_contour").sum()),
}
with open("/kaggle/working/apo_gray55_train_eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
